# 21 · AGENTS.md and Agent Skills
Level L3 · Part 4 · professional · ~30 min · builds on 05

## Goal
By the end you have an `AGENTS.md` for Trackr and a `triage-runbook` skill that `skills-ref` accepts. A real model answers a question about a late push notification twice: once with only the always-on rules (1 call), and once with the skill, where it asks for the runbook first and then answers (2 calls).

## Why it matters
Without these files, a team has two bad options. It can paste every rule into the system prompt, so every call pays for rules that most questions never use. Or the rule is missing, and the model guesses. Here is the cost of the first option with example numbers:

In [1]:
runbooks, chars_each = 30, 4_000   # example size: 30 team runbooks of 4,000 characters each
print("pasted into every call:", runbooks * chars_each, "characters")
print("needed for one issue:  ", chars_each, "characters")

pasted into every call: 120000 characters
needed for one issue:   4000 characters


`AGENTS.md` holds the rules that apply to every task in a folder, and the closest file wins. A skill holds a procedure for one kind of task. The model sees a one-line summary of each skill, and it loads the full text only when a task needs it. A rule that applies to every task belongs in `AGENTS.md`, not in a skill. A question that needs no team rule needs neither.

## The idea
- **AGENTS.md** is a plain Markdown file of project rules that a coding agent reads before it works. Where several files exist, the one closest to the code is used.
- **An Agent Skill** is a folder with a `SKILL.md` file. Its frontmatter (`name`, `description`) is always visible. The body, and any `scripts/`, `references/` or `assets/` files, load only when a task needs them. That order is *progressive disclosure*.
- **Skills over MCP** is the same folder served by an MCP server, so a host can discover skills from the server and read their files with `resources/read`.

Analogy: `AGENTS.md` is the house rules on the fridge. A skill is the recipe binder on the shelf. The fridge says the binder has a recipe for each dish. You open the binder only when you cook that dish.

## Picture
```mermaid
flowchart TD
  A[AGENTS.md: nearest file wins] --> P[System prompt: always-on rules]
  K[skills/triage-runbook: name + description] -->|catalogue line only| P
  P --> M[Model]
  M -->|asks: load_skill triage-runbook| H[Host reads the SKILL.md body]
  H --> M
  M --> R[Answer]
```

## Step by step

### 1. Write the Trackr rules and a nested AGENTS.md
The Trackr project lives in `../data/21/trackr`. The root `AGENTS.md` holds the team rules. The mobile service has its own file, which replaces the root file for code under `services/mobile`.

In [2]:
from pathlib import Path
ROOT = Path("../data/21/trackr")
Path(ROOT / "services/mobile").mkdir(parents=True, exist_ok=True)
Path(ROOT / "services/web").mkdir(parents=True, exist_ok=True)
print("project folder ready:", ROOT.resolve().name)

project folder ready: trackr


The root file holds the team rules for the whole project. Write it first.

In [3]:
%%writefile ../data/21/trackr/AGENTS.md
# Trackr: rules for coding agents
- Run the tests before you propose a change.
- Never close an issue without a reason in the comment.
- Use only issue ids that exist in Trackr.

Overwriting ../data/21/trackr/AGENTS.md


The mobile service keeps its own rules in its own folder. Its file is written next.

In [4]:
%%writefile ../data/21/trackr/services/mobile/AGENTS.md
# Mobile service rules
- Reproduce mobile bugs on the Android 14 test device first.
- Ship hotfixes behind a feature flag.
- Put the issue id in every commit message.

Overwriting ../data/21/trackr/services/mobile/AGENTS.md


### 2. Pick the nearest AGENTS.md
A host does not read every `AGENTS.md` in the repo. It starts at the file being edited and walks up. The first `AGENTS.md` it finds is the one it uses. Walking stops at the project root.

In [5]:
def nearest_agents_md(file: Path) -> Path | None:
    """Walk up from the file's folder; the first AGENTS.md found wins."""
    root = ROOT.resolve()
    for folder in file.resolve().parents:
        if (folder / "AGENTS.md").exists():
            return folder / "AGENTS.md"
        if folder == root:
            break
    return None

for edited in [ROOT / "services/mobile/push.py", ROOT / "services/web/app.py"]:
    found = nearest_agents_md(edited).relative_to(ROOT.resolve())
    print(f"{edited.relative_to(ROOT)} -> {found}")

SYSTEM = nearest_agents_md(ROOT / "services/mobile/push.py").read_text(encoding="utf-8")

services/mobile/push.py -> services/mobile/AGENTS.md
services/web/app.py -> AGENTS.md


The mobile file won for the push code. The web file has no nested rules, so the root file applies. The mobile file does not add to the root rules. It replaces them for that folder, which is why a team copies any rule it still needs into its own file.

### 3. Write the triage-runbook skill
A skill is a folder whose name matches the `name` field. Its body is the team runbook for one kind of task: a push delay. The frontmatter is what the model sees at all times.

In [6]:
Path(ROOT / "skills/triage-runbook").mkdir(parents=True, exist_ok=True)
SKILLS_DIR = (ROOT / "skills").resolve()
SKILL_DIR = SKILLS_DIR / "triage-runbook"
print("skill folder:", SKILL_DIR.relative_to(ROOT.resolve()))

skill folder: skills/triage-runbook


The frontmatter and the body are one file. The cell below writes `SKILL.md` into the skill folder.

In [7]:
%%writefile ../data/21/trackr/skills/triage-runbook/SKILL.md
---
name: triage-runbook
description: Triage a Trackr issue with the team runbook. Use when someone asks what to do about an issue, how urgent it is, or whether to page the on-call.
metadata:
  owner: ops-desk
  version: "1.0"
---

# Triage runbook

1. Read the issue title and severity in Trackr.
2. Find the symptom in the table and take its first steps.
3. Page the on-call only when the table says so. Say which rule you used.
4. Never close the issue in this step.

| Symptom | First steps | Page when |
|---|---|---|
| Push notifications arrive late | Check the push queue depth. Ask the device owner to re-register the push token. | More than 15 minutes late for 3 or more users: page the mobile on-call. |
| TLS certificate expires within 14 days | Renew with the infra CA. Check the chain on the edge. | Expires within 3 days: page the infra on-call. |

Overwriting ../data/21/trackr/skills/triage-runbook/SKILL.md


### 4. Read the catalogue line and the body
`skills_ref.read_properties` parses the frontmatter. The catalogue line the model always sees is the name and the description. The body is the rest of the file. Compare the two sizes.

In [8]:
from skills_ref import read_properties, validate
props = read_properties(SKILL_DIR)
body = (SKILL_DIR / "SKILL.md").read_text(encoding="utf-8").split("---", 2)[2]
catalogue_line = f"{props.name}: {props.description}"
print("catalogue line:", len(catalogue_line), "characters")
print("full body:     ", len(body), "characters")

catalogue line: 159 characters
full body:      632 characters


### 5. Validate the skill with skills-ref
`validate` returns a list of problems. An empty list means the skill is valid. A copy with an upper-case name shows the check that fails.

In [9]:
print("trackr skill:", validate(SKILL_DIR) or "valid")
bad = Path("../data/21/bad-skills/Triage-Runbook")
bad.mkdir(parents=True, exist_ok=True)
(bad / "SKILL.md").write_text((SKILL_DIR / "SKILL.md").read_text(encoding="utf-8").replace("name: triage-runbook", "name: Triage-Runbook"), encoding="utf-8")
print("bad copy:", validate(bad))

trackr skill: valid
bad copy: ["Skill name 'Triage-Runbook' must be lowercase"]


### 6. Serve the same skill over MCP
A server that publishes skills puts each `SKILL.md` at a `skill://` URI and reads it with `resources/read`. The in-process server below does that. The host then checks what it read: the SHA-256 digest and byte size must match the bytes on disk. The Skills extension manifest lists exactly these values. This lesson builds the listing entry by hand, because the `mcp` 2.3 SDK has no `skills/list` handler yet.

In [10]:
import hashlib, json
from mcp import Client
from mcp.server import MCPServer
skills_mcp = MCPServer("Trackr skills")

@skills_mcp.resource("skill://triage-runbook/SKILL.md", title="triage-runbook", mime_type="text/markdown")
def triage_runbook_skill() -> str:
    return (SKILL_DIR / "SKILL.md").read_text(encoding="utf-8")

async with Client(skills_mcp) as skills_client:
    listed = await skills_client.list_resources()
    print("resources:", [r.uri for r in listed.resources])
    got = await skills_client.read_resource("skill://triage-runbook/SKILL.md")
    served = got.contents[0].text.encode("utf-8")

on_disk = (SKILL_DIR / "SKILL.md").read_bytes()
entry = {"uri": "skill://triage-runbook/SKILL.md",
         "frontmatter": {"name": props.name, "description": props.description},
         "resources": [{"uri": "skill://triage-runbook/SKILL.md",
                        "digest": "sha256:" + hashlib.sha256(served).hexdigest(), "size": len(served)}]}
print("digest and size match the file on disk:", served == on_disk, entry["resources"][0]["size"])
print(json.dumps(entry, indent=2)[:400])

resources: ['skill://triage-runbook/SKILL.md']
digest and size match the file on disk: True 862
{
  "uri": "skill://triage-runbook/SKILL.md",
  "frontmatter": {
    "name": "triage-runbook",
    "description": "Triage a Trackr issue with the team runbook. Use when someone asks what to do about an issue, how urgent it is, or whether to page the on-call."
  },
  "resources": [
    {
      "uri": "skill://triage-runbook/SKILL.md",
      "digest": "sha256:c6678c14df3fded0193f6bba51e69373f2e5c3da


### 7. The model without the skill (`needs-llm`, call 1 of 3)
The model gets the always-on rules and the question. Nothing else. It has no way to know the team's push rule.

In [11]:
import sys
sys.path.insert(0, "..")   # so llm_setup and llm_replay import
import llm_replay
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
QUESTION = "Issue 7, 'Push notifications arrive late', has been 20 minutes late for 4 users. What do I do now?"
llm = None   # filled by the model cell; stays None if that cell is skipped
print("question ready; the model cells below need a model (or are skipped)")

question ready; the model cells below need a model (or are skipped)


The first model cell. `get_llm` records the call the first time and replays it after that.

In [12]:
import logging
for noisy in ["httpx", "google_genai", "vcr"]:   # their INFO lines print the model URL and project id
    logging.getLogger(noisy).setLevel(logging.WARNING)
from llm_setup import get_llm
llm = get_llm("21_agents-md-and-skills", max_calls=3)
plain = await llm.ainvoke([SystemMessage(SYSTEM), HumanMessage(QUESTION)])
print("without the skill:", plain.text)

provider   key / server                       model
gemini     ✓ GOOGLE_CLOUD_API_KEY set         gemini-3.1-flash-lite  recorded <- chosen
                                              gemini-3.5-flash-lite  
                                              gemini-3.8-flash  
anthropic  ✗ ANTHROPIC_API_KEY not set        claude-haiku-5-5  
                                              claude-sonnet-5-5  
                                              claude-opus-5-5  
                                              claude-fable-5-1  
openai     ✗ OPENAI_API_KEY not set           gpt-6-luna  
                                              gpt-6.1-sol  
                                              gpt-6-astra  
ollama     ✗ Ollama not running               qwen2.5:7b  


LLM mode: replay (gemini__gemini-3.1-flash-lite.yaml)
Using gemini:gemini-3.1-flash-lite


without the skill: To address **Issue 7: Push notifications arrive late**, please follow the established mobile service protocol:

### 1. Reproduce on Android 14
Before attempting a fix, you must verify the latency on the **Android 14 test device**. 
*   Check if the delay is related to Android 14’s "Doze" mode or background execution restrictions.
*   Verify if the issue persists across different network conditions (Wi-Fi vs. Cellular) on the test device.

### 2. Develop and Ship a Hotfix
Once the root cause is identified and the fix is implemented:
*   **Feature Flag:** Wrap the fix in a feature flag (e.g., `enable_optimized_push_delivery`). Do not push the fix directly to production without the flag.
*   **Commit Message:** Ensure every commit related to this fix includes the issue ID. 
    *   *Example:* `git commit -m "ISSUE-7: Implement priority queue for push notifications"`

### 3. Deployment Strategy
*   **Rollout:** Enable the feature flag for a small subset of users (canary 

### 8. The skill is loaded only on request (`needs-llm`, call 2 of 3)
Now the system prompt also has the catalogue line, not the body. A tool named `load_skill` returns the body. The model asks for it, and the host runs the tool. The body is not in the prompt yet.

In [13]:
@tool
def load_skill(name: str) -> str:
    """Load the full instructions of a skill from the catalogue. Pass the exact skill name."""
    try:
        return (SKILLS_DIR / name / "SKILL.md").read_text(encoding="utf-8")
    except FileNotFoundError:
        # report the path relative to the project, not the absolute folder on this machine
        raise FileNotFoundError(f"skills/{name}/SKILL.md (no such skill in the catalogue)") from None

CATALOGUE = f"Skills you can load (name: description):\n- {props.name}: {props.description}\nCall load_skill with the exact name before you follow a skill."
WITH_SKILL = SYSTEM + "\n\n" + CATALOGUE
print("load_skill ready:", load_skill.name, "| catalogue lines in prompt:", CATALOGUE.count("\n- "))

load_skill ready: load_skill | catalogue lines in prompt: 1


The catalogue line goes into the prompt, the body does not. The model now returns a tool call instead of an answer.

In [14]:
asks = await llm.bind_tools([load_skill]).ainvoke([SystemMessage(WITH_SKILL), HumanMessage(QUESTION)])
print("model asks for:", [(c["name"], c["args"]) for c in asks.tool_calls])

model asks for: [('load_skill', {'name': 'triage-runbook'})]


### 9. The host runs the tool and the model answers (`needs-llm`, call 3 of 3)
The host runs each requested tool and sends the body back as a tool message. The model then writes its answer from the body.

In [15]:
results = [await load_skill.ainvoke(c) for c in asks.tool_calls]
answer = await llm.bind_tools([load_skill]).ainvoke([SystemMessage(WITH_SKILL), HumanMessage(QUESTION), asks, *results])
print("with the skill:", answer.text)

with the skill: Based on the triage runbook, you should **page the mobile on-call** immediately.

The issue "Push notifications arrive late" meets the criteria for paging because it has been 20 minutes late for 4 users (the threshold is more than 15 minutes late for 3 or more users).

**Next steps:**
1. Page the mobile on-call.
2. Check the push queue depth.
3. Ask the affected users to re-register their push tokens.
4. Remember to include the issue ID (7) in any related commit messages if you implement a fix.


Compare the two answers. The first one never mentions paging or the 15-minute, 3-user threshold, because the model had no way to know the team rule. The second one quotes the runbook: it applies the push row, the 15-minute, 3-user threshold, and it names the mobile on-call to page. The cost was 2 model calls instead of 1, plus the 632-character body, which entered the prompt only when the model asked for it.

Without a model, the cell below still shows what the host would send: the body that `load_skill` returns for the same name.

In [16]:
if llm is None:   # fallback: no model here, so only the tool side runs
    body_text = load_skill.invoke({"name": "triage-runbook"})
    push_row = [line for line in body_text.splitlines() if "Push notifications" in line][0]
    print("no model here. The host would send this body on request. Its push row:", push_row)

## At work
- Monorepos keep one `AGENTS.md` per service, and the closest one wins. A stale root file then hides a team's rule without any error. Teams check these files in CI, the way they test code → lesson 27.
- Every skill name and description is in context on every call. Large catalogues cost tokens on each request, so teams keep descriptions short and bodies on demand → lesson 24.
- A skill is text the model obeys. A skill from another team, or from a shared server, can carry instructions that do harm. Hosts ask the user before they load a skill, and they check digests before they use it → lesson 29.

## What just happened
- The host chose the nearest `AGENTS.md` for the file it edits. The mobile file replaced the root file for that folder.
- `skills-ref` checked the frontmatter and the folder name. The bad copy failed on its upper-case name.
- The same bytes came back over `resources/read` as from disk, so the digest matched.
- The model saw the catalogue line first. It asked for the body, and only then did it have the team's push rule.
- The comparison cost 2 model calls with the skill instead of 1 without it. The 632-character body entered the prompt only when the model asked for it.

## Common mistakes
Asking for a skill by a name that is not in the catalogue. The catalogue says `triage-runbook`. `triage_runbook` with an underscore is another folder, so the tool fails:

In [17]:
load_skill.invoke({"name": "triage_runbook"})

FileNotFoundError: skills/triage_runbook/SKILL.md (no such skill in the catalogue)

Fix: pass the exact `name` from the catalogue line. The validator would also have caught a name that does not match its folder.

## Try it
Copy the `triage-runbook` folder to `triage-runbook-copy` without changing the `name` line. Run `validate` on the copy and print the result. Then fix the name so it matches the folder, and validate again. No model is needed.

In [18]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [19]:
import shutil
copy = SKILLS_DIR / "triage-runbook-copy"
shutil.copytree(SKILL_DIR, copy, dirs_exist_ok=True)
print("before:", validate(copy))
text = (copy / "SKILL.md").read_text(encoding="utf-8").replace("name: triage-runbook", "name: triage-runbook-copy", 1)
(copy / "SKILL.md").write_text(text, encoding="utf-8")
print("after:", validate(copy))

before: ["Directory name 'triage-runbook-copy' must match skill name 'triage-runbook'"]
after: []


## Key terms
- **AGENTS.md** — a Markdown file of project rules for coding agents. The closest file to the code wins.
- **Agent Skill** — a folder with a `SKILL.md` file whose frontmatter is always visible and whose body loads on request.
- **SKILL.md frontmatter** — the YAML block at the top of `SKILL.md`. `name` and `description` are required.
- **progressive disclosure** — loading metadata first, the body when a task needs it, and other files only when they are used.
- **skill catalogue** — the list of name and description lines the model sees on every call.
- **Skills over MCP** — the MCP Skills extension (`io.modelcontextprotocol/skills`). A server lists skills with `skills/list`, and the files are read with `resources/read`.
- **skills-ref** — the reference library and `skills-ref validate` command that check a skill folder against the spec.

Close the recorder, which saves the recording if this run recorded.

In [20]:
llm_replay.stop()
print("lesson 21 done: AGENTS.md picks the rules, the skill loads on request, the same skill is served over MCP")

lesson 21 done: AGENTS.md picks the rules, the skill loads on request, the same skill is served over MCP
